In [1]:
from pymongo import MongoClient
from collections import defaultdict
from datetime import datetime


In [11]:
# MongoDB connection
client = MongoClient("mongodb://localhost:27017/")
db = client["Retail_Business"]

orders_raw = db["Orders"]
customers_wh = db["customers_warehouse"]  # WAREHOUSE layer


In [12]:
# Load all raw orders into memory
raw_orders = list(orders_raw.find())

print(f"Total raw documents loaded: {len(raw_orders)}")


Total raw documents loaded: 200000


In [13]:
customers = defaultdict(lambda: {
    "customer_id": None,
    "customer_name": None,
    "gender": None,
    "city": None,
    "total_spent": 0.0,
    "total_orders": set(),
    "products": defaultdict(int),
    "branches": defaultdict(int),
    "monthly_spending": defaultdict(float)
})


In [14]:
for doc in raw_orders:
    cust_id = doc["customer"]["id"]
    sale_id = doc["sale"]["id"]

    customer = customers[cust_id]

    # Basic customer info
    customer["customer_id"] = cust_id
    customer["customer_name"] = doc["customer"]["name"]
    customer["gender"] = doc["customer"]["gender"]
    customer["city"] = doc["customer"]["city"]

    # Orders & total spending (avoid duplicate sales)
    if sale_id not in customer["total_orders"]:
        customer["total_orders"].add(sale_id)
        customer["total_spent"] += doc["sale"]["total_amount"]

    # Products aggregation
    product_name = doc["product"]["name"]
    customer["products"][product_name] += doc["product"]["quantity"]

    # Branch preference
    branch_name = doc["branch"]["name"]
    customer["branches"][branch_name] += 1

    # Monthly spending
    sale_date = datetime.strptime(doc["sale"]["date"], "%Y-%m-%d")
    month_key = sale_date.strftime("%Y-%m")
    customer["monthly_spending"][month_key] += doc["sale"]["total_amount"]


In [15]:
final_docs = []

for customer in customers.values():
    final_doc = {
        "customer_id": customer["customer_id"],
        "customer_name": customer["customer_name"],
        "gender": customer["gender"],
        "city": customer["city"],
        "total_spent": round(customer["total_spent"], 2),
        "total_orders": len(customer["total_orders"]),
        "top_products": sorted(
            [{"product_name": k, "quantity": v} for k, v in customer["products"].items()],
            key=lambda x: x["quantity"],
            reverse=True
        )[:5],
        "preferred_branch": max(customer["branches"], key=customer["branches"].get),
        "monthly_spending": dict(customer["monthly_spending"])
    }

    final_docs.append(final_doc)

print(f"Total customers aggregated: {len(final_docs)}")


Total customers aggregated: 18278


In [16]:
# Clear old warehouse data
customers_wh.delete_many({})

# Insert new warehouse data
customers_wh.insert_many(final_docs)

print("✅ customers_warehouse built successfully")


✅ customers_warehouse built successfully


In [17]:
# Preview one warehouse document
customers_wh.find_one()


{'_id': ObjectId('6979223f54d2c50afb315a5b'),
 'customer_id': 3649,
 'customer_name': 'Sara Mahmoud',
 'gender': 'Female',
 'city': 'Ismailia',
 'total_spent': 14991.11,
 'total_orders': 2,
 'top_products': [{'product_name': 'Home Appliances Product 4003',
   'quantity': 5},
  {'product_name': 'Beauty Product 19887', 'quantity': 4},
  {'product_name': 'Electronics Product 28176', 'quantity': 4},
  {'product_name': 'Beauty Product 19951', 'quantity': 4},
  {'product_name': 'Electronics Product 35551', 'quantity': 3}],
 'preferred_branch': 'Mansoura Main Branch',
 'monthly_spending': {'2023-07': 11246.67, '2022-04': 89937.76}}